# Min-Max vs Z-Score Normalisation

**DS4DH Practice Pack · Module 08 — Index and Metric Design**

*Technique:* Making units disappear, and inverting polarity so high always means the same thing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/08a_normalization.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

To combine income (dollars), housing burden (percent) and population (people)
into one number, the units have to go. There are two standard ways, and they
answer different questions:

- **Min-max** → "where does this place sit between the worst and best case?"
  Bounded [0, 1]. Every value depends on the two extremes.
- **Z-score** → "how many standard deviations from average is this place?"
  Unbounded. Depends on the mean and spread, not on the extremes.

Neither is correct in general. The choice is a modelling decision you have to be
able to defend.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
FEATURES = ['Total', 'renter_owner_gap', 'tot_income', 'log_pop']
RAW_NEEDED = ['Total', 'renter_owner_gap', 'tot_income', 'tot_pop']

feat = base.dropna(subset=RAW_NEEDED).copy()
feat['log_pop'] = np.log10(feat['tot_pop'])

print(f'{len(base)} CSDs -> {len(feat)} with complete data on all four columns')
print(f'{len(base) - len(feat)} dropped (census suppression in small places)')
print()
for city, n in feat['cma'].value_counts().items():
    print(f'  {city:<11} {n:>3} CSDs')

In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

def zscore(s):
    return (s - s.mean()) / s.std()

norm = pd.DataFrame({
    'raw': feat['tot_income'],
    'minmax': minmax(feat['tot_income']),
    'z': zscore(feat['tot_income']),
})
print(norm.describe().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, title in zip(axes, ['raw', 'minmax', 'z'],
                          ['Raw income ($)', 'Min-max [0,1]', 'Z-score']):
    ax.hist(norm[col], bins=25, edgecolor='white', linewidth=0.5)
    ax.set_title(title)
plt.tight_layout()
plt.show()

print('The shape is identical in all three. Normalisation moves and rescales a')
print('distribution; it does not reshape it. Skew survives both transforms.')

## The outlier sensitivity that decides it

Min-max divides by the range, so a single extreme value compresses everything
else into a narrow band. Z-score is affected too — the mean and sd both move —
but far less dramatically.

In [ ]:
spiked = feat['tot_income'].copy()
spiked.iloc[0] = spiked.max() * 4          # one implausibly rich CSD

print(f'{"":<28}{"original":>12}{"with outlier":>15}')
print('-' * 55)
mm_o, mm_s = minmax(feat['tot_income']), minmax(spiked)
z_o, z_s = zscore(feat['tot_income']), zscore(spiked)
print(f'{"min-max: median value":<28}{mm_o.median():>12.3f}{mm_s.median():>15.3f}')
print(f'{"min-max: IQR":<28}{mm_o.quantile(.75) - mm_o.quantile(.25):>12.3f}'
      f'{mm_s.quantile(.75) - mm_s.quantile(.25):>15.3f}')
print(f'{"z-score: median value":<28}{z_o.median():>12.3f}{z_s.median():>15.3f}')
print(f'{"z-score: IQR":<28}{z_o.quantile(.75) - z_o.quantile(.25):>12.3f}'
      f'{z_s.quantile(.75) - z_s.quantile(.25):>15.3f}')
print()
print('One bad value halves the min-max spread of every other CSD.')

### 🔧 Your turn 1

Change the multiplier from `* 4` to `* 1.5`, then `* 20`.

At what point does min-max become unusable? Since census data does contain
genuine extremes, what does that imply for an index built on min-max?

## Polarity

Some components mean "worse" when high (housing burden) and some mean "better"
when high (income). Combining them without fixing that produces an index where a
high score means nothing consistent.

For min-max, invert with `1 − x`. For z-scores, negate.

In [ ]:
comp = pd.DataFrame({
    'burden': minmax(feat['Total']),          # high = worse
    'income': minmax(feat['tot_income']),     # high = better
})
comp['income_inverted'] = 1 - comp['income']  # now high = worse

naive = (comp['burden'] + comp['income']) / 2
fixed = (comp['burden'] + comp['income_inverted']) / 2

check = feat.assign(naive=naive.values, fixed=fixed.values)
print('Highest scores under each version:')
print()
for name in ['naive', 'fixed']:
    top = check.nlargest(3, name)[['geography_name', 'Total', 'tot_income', name]]
    print(f'--- {name} ---')
    print(top.to_string(index=False))
    print()

In [ ]:
print(f'correlation between the two versions: {naive.corr(fixed):.3f}')
print()
print('The naive index scores a place highly for being BOTH heavily burdened')
print('AND rich — two things that do not belong on the same end of a scale.')
print('The corrected one is a coherent "housing stress" direction.')

### 🔧 Your turn 2

Build the same pair of indices using z-scores instead of min-max (negate rather
than `1 − x`).

Do the top three places change? Which version would you defend to a housing
agency, and what one sentence explains the choice?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** By `* 20` min-max is unusable — almost every real CSD is
squeezed below 0.1 and the index can no longer distinguish them. Since census
income data genuinely contains extreme municipalities, an index built on raw
min-max inherits that fragility. The standard defences are to winsorise at the 5th
and 95th percentiles before scaling, or to use z-scores, or to transform first
(as `log_pop` does). What you must not do is use min-max and not mention it.

**Your turn 2.** The top three usually differ. Z-scores let a place with one
extreme component dominate, because they are unbounded; min-max caps each
component's contribution at 1. For a housing agency the defensible sentence is
something like: *"Components were min-max scaled after winsorising at the 5th and
95th percentiles, so that no single indicator can dominate the composite and the
index stays bounded on [0, 1]."* Any choice is fine if you can write that
sentence; none is fine if you cannot.

</details>

## Where this stops

The components are now comparable and pointing the same way. Combining them is
the next notebook — and the weights are a bigger decision than the scaling was.